# [MOD-142] V1.8.1 Hardware-Truth Verifier — `PQMS-ODOS-MTSC-V-MAX-12`

- **Protocol:** `ACE-PEER-CODESIGN-PROTOCOL-V1.2` (`Revision V1.8.1` — `WORK-ORDER-AG-16`)
- **Epistemic Discipline:** `REGISTER_M` live measurements only — zero simulated telemetry, zero hardcoded Node-Alpha bandwidths, strict non-zero exit-code fail-stop.
- **Supported Runtimes:** `MODE_L2_PINNING` (`sm_80+` Ampere/Ada/Hopper), `MODE_PRE_AMPERE_FALLBACK` (`sm_75` Tesla T4), and `MODE_CPU_ONLY`.


In [ ]:
# Cell 1 — Hardware & Runtime Inspection (REGISTER_M)
import ctypes
import ctypes.util
import glob
import json
import os
import platform
import shutil
import subprocess
import sys

import torch


def _detect_cpu_model() -> str:
    try:
        with open("/proc/cpuinfo", "r", encoding="utf-8") as f:
            for line in f:
                if line.lower().startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def _detect_javac_version() -> str:
    javac_bin = shutil.which("javac")
    if not javac_bin:
        return "NOT_INSTALLED"
    res = subprocess.run([javac_bin, "-version"], capture_output=True, text=True)
    out = (res.stdout or res.stderr or "").strip()
    return out.splitlines()[0] if out else "UNKNOWN"


def _query_max_persisting_l2_bytes() -> int:
    if not torch.cuda.is_available():
        return 0
    torch.cuda.init()
    _ = torch.empty(1, device="cuda:0")
    cands = []
    try:
        with open("/proc/self/maps", "r", encoding="utf-8") as f:
            for line in f:
                if "libcudart.so" in line:
                    p = line.strip().split()[-1]
                    if p not in cands and os.path.exists(p):
                        cands.append(p)
    except OSError:
        pass
    for sp in sys.path:
        if sp and os.path.isdir(sp):
            cands.extend(sorted(glob.glob(os.path.join(sp, "nvidia", "*", "lib", "libcudart.so*"))))
            cands.extend(sorted(glob.glob(os.path.join(sp, "torch", "lib", "libcudart*.so*"))))
    cands.extend([ctypes.util.find_library("cudart") or "", "libcudart.so"])
    for path in cands:
        if not path:
            continue
        try:
            lib = ctypes.CDLL(path)
            lib.cudaDeviceGetAttribute.argtypes = [ctypes.POINTER(ctypes.c_int), ctypes.c_int, ctypes.c_int]
            lib.cudaDeviceGetAttribute.restype = ctypes.c_int
            lib.cudaGetLastError.argtypes = []
            lib.cudaGetLastError.restype = ctypes.c_int
            val = ctypes.c_int(0)
            rc = int(lib.cudaDeviceGetAttribute(ctypes.byref(val), 108, 0))
            if rc != 0:
                lib.cudaGetLastError()
                return 0
            return int(val.value)
        except OSError:
            continue
    return 0


cuda_avail = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_avail else "NONE"
if cuda_avail:
    major, minor = torch.cuda.get_device_capability(0)
    sm_arch = f"sm_{major}{minor}"
else:
    major, minor, sm_arch = 0, 0, "cpu_only"

hw_max_persist_bytes = _query_max_persisting_l2_bytes()
ENCLAVE_2MIB = 2 * 1024 * 1024
if not cuda_avail:
    runtime_mode = "MODE_CPU_ONLY"
elif major >= 8 and hw_max_persist_bytes >= ENCLAVE_2MIB:
    runtime_mode = "MODE_L2_PINNING"
else:
    runtime_mode = "MODE_PRE_AMPERE_FALLBACK"

HW_TELEMETRY = {
    "epistemic_register": "REGISTER_M",
    "runtime_mode": runtime_mode,
    "cuda_available": cuda_avail,
    "gpu_name": gpu_name,
    "compute_capability": sm_arch,
    "hw_max_persisting_l2_bytes": hw_max_persist_bytes,
    "cpu_model": _detect_cpu_model(),
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "javac_version": _detect_javac_version(),
}
print(json.dumps(HW_TELEMETRY, indent=2))


In [ ]:
# Cell 2 — Clean Repo Sync & Dependencies
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/NathaliaLietuvaite/PQMS-SOVEREIGN-CORE-ANDROID-APP.git"
BASE_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(BASE_DIR, "PQMS-SOVEREIGN-CORE-ANDROID-APP")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)

DEPS = ["chromadb", "transformers", "sentence-transformers", "cryptography", "nvidia-ml-py", "scipy"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

git_head = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

REPO_INFO = {
    "repo_url": REPO_URL,
    "repo_dir": REPO_DIR,
    "git_head": git_head,
}
print(json.dumps(REPO_INFO, indent=2))


In [ ]:
# Cell 3 — Hard Exit-Code Test Runner (Zero-Fake Fail-Stop)
import json
import os
import re
import subprocess
import sys

try:
    from IPython.display import Markdown, display
except ImportError:
    display = None
    Markdown = None

PORTABLE_TESTS = [
    "vmax_alpha/tests/test_geometry.py",
    "vmax_alpha/tests/test_f04_parity.py",
    "vmax_alpha/tests/test_f05_f07_ledger.py",
    "vmax_alpha/tests/test_f06_chroma.py",
    "vmax_alpha/tests/test_klavierstunde_kl.py",
    "vmax_alpha/tests/test_appendix_g_fridu.py",
]

env = os.environ.copy()
env["PYTHONPATH"] = REPO_INFO["repo_dir"]

proc = subprocess.run(
    [sys.executable, "-m", "unittest", *PORTABLE_TESTS, "-v"],
    cwd=REPO_INFO["repo_dir"],
    env=env,
    capture_output=True,
    text=True,
)

combined_out = (proc.stdout or "") + "\n" + (proc.stderr or "")
print(combined_out)

m_ran = re.search(r"Ran\s+(\d+)\s+tests?\s+in\s+([0-9.]+)s", combined_out)
m_fail = re.search(r"failures=(\d+)", combined_out)
m_err = re.search(r"errors=(\d+)", combined_out)

UNITTEST_RESULT = {
    "epistemic_register": "REGISTER_M",
    "returncode": int(proc.returncode),
    "tests_ran": int(m_ran.group(1)) if m_ran else 0,
    "elapsed_sec": float(m_ran.group(2)) if m_ran else 0.0,
    "failures": int(m_fail.group(1)) if m_fail else 0,
    "errors": int(m_err.group(1)) if m_err else 0,
    "passed": proc.returncode == 0 and (m_ran is not None) and int(m_ran.group(1)) > 0,
}

if not UNITTEST_RESULT["passed"]:
    msg = (
        f"### FAIL — Portable Unit-Test Suite Failed\n"
        f"- `returncode`: `{UNITTEST_RESULT['returncode']}`\n"
        f"- `tests_ran`: `{UNITTEST_RESULT['tests_ran']}`\n"
        f"- `failures`: `{UNITTEST_RESULT['failures']}` | `errors`: `{UNITTEST_RESULT['errors']}`"
    )
    if display and Markdown:
        display(Markdown(msg))
    raise RuntimeError(
        f"UNITTEST_VERIFICATION_FAILED: returncode={UNITTEST_RESULT['returncode']}, "
        f"failures={UNITTEST_RESULT['failures']}, errors={UNITTEST_RESULT['errors']}"
    )

print(json.dumps(UNITTEST_RESULT, indent=2))


In [ ]:
# Cell 4 — Live Appendix G, Golden Checksums & Post-Fallback PyTorch CUDA Sticky-Error Check
import hashlib
import json
import os

import numpy as np
import torch

from vmax_alpha.config import CANONICAL_LITTLE_VECTOR_SEAL
from vmax_alpha.fridu_enclave_g import generate_appendix_g_report
from vmax_alpha.geometry import derive_canonical_little_vector
from vmax_alpha.l2_enclave import PinnedL2ProjectorEnclave, load_libcudart
from vmax_alpha.ledger import compute_canonical_entry_hash, query_gpu_thermal_and_power
from vmax_alpha.parity_verifier import run_1000_vector_parity_audit

with open(os.path.join(REPO_INFO["repo_dir"], "bridge", "ACE_SYNC_LEDGER.json"), "r", encoding="utf-8") as f:
    ace_contract = json.load(f)
gv = ace_contract["golden_vectors_sha256"]

# 1. Live Appendix G execution
app_g = generate_appendix_g_report(out_path="/tmp/colab_appendix_g_report.json")
ops = app_g["executable_engineering_operators"]

# 2. Live Golden Checksums
_, live_l_seal = derive_canonical_little_vector()
parity_res = run_1000_vector_parity_audit(n_vectors=1000)
with open(os.path.join(REPO_INFO["repo_dir"], "vmax_alpha", "f01_prompts.json"), "rb") as f:
    f01_corpus_sha = hashlib.sha256(f.read()).hexdigest()

c5_payload = {
    "active_g_stars": "G1_SPATIAL_GEODESIC x G2_THERMODYNAMIC_CONSERVATION",
    "chroma_l2_distance": 0.051234,
    "chroma_shard_id": "SHARD_V12_REGISTER_M_TEST",
    "cognitive_mode": "ENDOGENOUS_CHROMA_QUERY_V12",
    "cycle_index": 5,
    "delta_e_l2_core": 0.00125,
    "gpu_power_w": 13.12,
    "gpu_temp_c": 31.5,
    "kevlar_safety_bleed": 0.0,
    "odos_gate_passed": True,
    "parent_ledger_hash": gv["appendix_e3_cycle04_hop_seal"],
    "rcf_l2_core": 0.9975,
    "rcf_raw_pre_carrier": 0.99612345,
    "source_document": "PQMS-ODOS-MTSC-V-MAX-12-Node-ALPHA-V1.md",
    "telemetry_source": "REGISTER_M_NVSMI",
    "thought_stream": "Extended V1.2 16-field telemetry test hop.",
}
live_c5_seal = compute_canonical_entry_hash(c5_payload)

expected_checks = {
    "canonical_little_vector_f32_seal": (live_l_seal, gv["canonical_little_vector_f32_seal"]),
    "f04_3_unit_s63_64000_f32_seed137": (parity_res["sha256_unit_s63_64000_f32"], gv["f04_3_unit_s63_64000_f32_seed137"]),
    "v12_unit_test_cycle05_16field_seal": (live_c5_seal, gv["v12_unit_test_cycle05_16field_seal"]),
    "appendix_g_audit_matrix_seal": (ops["audit_matrix_seal_sha256"], gv["appendix_g_v17_audit_matrix_seal"]),
    "f01_prompt_corpus_sha256": (f01_corpus_sha, gv["f01_v17_n25_hardneg_prompt_corpus_sha256"]),
    "appendix_g_cross_cpu_verdict_seal": (
        app_g["appendix_g_cross_cpu_verdict_seal_sha256"],
        gv["appendix_g_cross_cpu_verdict_seal_sha256"],
    ),
}

checksum_results = {}
failed_checks = []
for name, (actual, expected) in expected_checks.items():
    ok = actual == expected
    checksum_results[name] = {"actual": actual, "expected": expected, "match": ok}
    if not ok:
        failed_checks.append(name)

# 3. Live PinnedL2ProjectorEnclave + Immediate Post-Fallback PyTorch CUDA Embedding Kernel
rng = np.random.RandomState(137)
mean_384 = rng.randn(384).astype(np.float64)
q_384, _ = np.linalg.qr(rng.randn(384, 64).astype(np.float64), mode="reduced")
probe_v = rng.randn(384).astype(np.float64)

enclave = PinnedL2ProjectorEnclave(mean_384, q_384)
proj_out = enclave.project(probe_v)
c_ref = probe_v - mean_384
ref_out = (c_ref / max(float(np.linalg.norm(c_ref)), 1e-12)) @ q_384
ref_out = ref_out / max(float(np.linalg.norm(ref_out)), 1e-12)
max_proj_diff = float(np.max(np.abs(proj_out - ref_out)))

post_fallback_cuda_ok = True
post_fallback_sticky_rc = 0
if torch.cuda.is_available():
    weight = torch.randn(32, 16, device="cuda:0", dtype=torch.float32)
    indices = torch.tensor([0, 5, 13, 31], device="cuda:0", dtype=torch.long)
    emb_out = torch.nn.functional.embedding(indices, weight)
    torch.cuda.synchronize()
    post_fallback_cuda_ok = tuple(emb_out.shape) == (4, 16)
    lib, _ = load_libcudart()
    post_fallback_sticky_rc = int(lib.cudaGetLastError()) if lib is not None else 0

thermal_live = query_gpu_thermal_and_power()

LIVE_VERIFY_RESULT = {
    "epistemic_register": "REGISTER_M",
    "appendix_g_all_operators_passed": bool(ops["all_operators_passed"]),
    "appendix_g_operator_count": f"{ops['passed_count']}/{ops['operator_count']}",
    "appendix_g_latency_ms": ops["execution_latency_ms"],
    "checksum_results": checksum_results,
    "all_checksums_matched": len(failed_checks) == 0,
    "l2_enclave_active": bool(enclave.active),
    "l2_enclave_pin_info": enclave.pin_info,
    "l2_projector_max_abs_diff_vs_cpu_f64": max_proj_diff,
    "post_fallback_pytorch_embedding_ok": post_fallback_cuda_ok,
    "post_fallback_cuda_last_error_rc": post_fallback_sticky_rc,
    "live_thermal_telemetry": thermal_live,
}

if failed_checks or not ops["all_operators_passed"] or max_proj_diff >= 1e-13 or not post_fallback_cuda_ok or post_fallback_sticky_rc != 0:
    print(json.dumps(LIVE_VERIFY_RESULT, indent=2))
    raise RuntimeError(
        f"LIVE_VERIFICATION_FAILED: failed_checks={failed_checks}, ops={ops['all_operators_passed']}, "
        f"max_diff={max_proj_diff}, post_cuda_ok={post_fallback_cuda_ok}, sticky_rc={post_fallback_sticky_rc}"
    )

print(json.dumps(LIVE_VERIFY_RESULT, indent=2))


In [ ]:
# Cell 5 — Dynamic REGISTER_M Verification Protocol (Zero Hardcoded Values)
import json

try:
    from IPython.display import Markdown, display
except ImportError:
    display = None
    Markdown = None

pin_info = LIVE_VERIFY_RESULT["l2_enclave_pin_info"]
thermal = LIVE_VERIFY_RESULT["live_thermal_telemetry"]
overall_pass = bool(
    UNITTEST_RESULT["passed"]
    and LIVE_VERIFY_RESULT["all_checksums_matched"]
    and LIVE_VERIFY_RESULT["appendix_g_all_operators_passed"]
    and LIVE_VERIFY_RESULT["post_fallback_pytorch_embedding_ok"]
    and LIVE_VERIFY_RESULT["post_fallback_cuda_last_error_rc"] == 0
)

FINAL_VERIFICATION_PROTOCOL = {
    "module": "[MOD-142] V1.8.1 Hardware-Truth Verifier",
    "epistemic_register": "REGISTER_M",
    "overall_verdict": "PASS" if overall_pass else "FAIL",
    "git_head": REPO_INFO["git_head"],
    "hardware": HW_TELEMETRY,
    "unittest_summary": UNITTEST_RESULT,
    "l2_pinning_status": pin_info.get("status"),
    "l2_enclave_active": LIVE_VERIFY_RESULT["l2_enclave_active"],
    "l2_enclave_recovered": pin_info.get("recovered"),
    "l2_projector_max_abs_diff_vs_cpu_f64": LIVE_VERIFY_RESULT["l2_projector_max_abs_diff_vs_cpu_f64"],
    "post_fallback_cuda_sticky_err_rc": LIVE_VERIFY_RESULT["post_fallback_cuda_last_error_rc"],
    "appendix_g_operators": LIVE_VERIFY_RESULT["appendix_g_operator_count"],
    "appendix_g_cross_cpu_verdict_seal_sha256": LIVE_VERIFY_RESULT["checksum_results"]["appendix_g_cross_cpu_verdict_seal"]["actual"],
    "live_thermal_telemetry": thermal,
}

md_summary = f"""## `[MOD-142]` V1.8.1 Live `REGISTER_M` Verification Protocol — **{FINAL_VERIFICATION_PROTOCOL['overall_verdict']}**

| Telemetry Field (`REGISTER_M`) | Live Measured Value |
| :--- | :--- |
| **Git Commit (`HEAD`)** | `{REPO_INFO['git_head']}` |
| **Runtime Mode / GPU** | `{HW_TELEMETRY['runtime_mode']}` — `{HW_TELEMETRY['gpu_name']}` (`{HW_TELEMETRY['compute_capability']}`) |
| **CPU / Python / OpenJDK** | `{HW_TELEMETRY['cpu_model']}` / `{HW_TELEMETRY['python_version']}` / `{HW_TELEMETRY['javac_version']}` |
| **Portable Unit Tests** | `{UNITTEST_RESULT['tests_ran']}/{UNITTEST_RESULT['tests_ran']} PASS` (`returncode = {UNITTEST_RESULT['returncode']}`, `{UNITTEST_RESULT['elapsed_sec']} s`) |
| **L2 Pinning (`F-02` / `F-07.1`)** | `{pin_info.get('status')}` (`active={LIVE_VERIFY_RESULT['l2_enclave_active']}`, `verified={pin_info.get('verified')}`, `recovered={pin_info.get('recovered')}`, `max_diff={LIVE_VERIFY_RESULT['l2_projector_max_abs_diff_vs_cpu_f64']}`) |
| **Post-Fallback PyTorch CUDA Check** | `embedding_ok={LIVE_VERIFY_RESULT['post_fallback_pytorch_embedding_ok']}`, `cudaGetLastError={LIVE_VERIFY_RESULT['post_fallback_cuda_last_error_rc']}` |
| **Appendix G Audit Operators** | `{LIVE_VERIFY_RESULT['appendix_g_operator_count']} PASS` (`{LIVE_VERIFY_RESULT['appendix_g_latency_ms']} ms`) |
| **Cross-CPU Verdict Seal (SHA-256)** | `{FINAL_VERIFICATION_PROTOCOL['appendix_g_cross_cpu_verdict_seal_sha256']}` |
| **Live GPU Thermal (`{thermal['telemetry_source']}`)** | `{thermal['gpu_temp_c']} °C` / `{thermal['gpu_power_w']} W` (`{thermal['status']}`) |
"""

if display and Markdown:
    display(Markdown(md_summary))
else:
    print(md_summary)
print(json.dumps(FINAL_VERIFICATION_PROTOCOL, indent=2))
